将数据转换为csv/parquet（专门为大数据处理设计，节省空间and计算高效）格式，同时将type列int64类型转换为int8节省空间

In [20]:
import numpy as np
import pandas as pd

In [21]:
# 对type进行映射转换
id2type=['clicks','carts','orders']
type2id={a:i for i,a in enumerate(id2type)}

In [22]:
pd.to_pickle(id2type,'../data/processData/id2type.pkl')
pd.to_pickle(type2id,'../data/processData/type2id.pkl')

In [29]:
def json_df(fn):
    """
    数据较大，分块读取
    :param fn: 数据集地址
    :return: DataFrame (session_id,aid,ts,type)
    """
    sessions=[]
    aids=[]
    tss=[]
    types=[]

    # lines设置按行读取，每次读取chunksize行数据，
    chunks=pd.read_json(fn,lines=True,chunksize=100_000)
    for chunk in chunks:
        for row_idx,session_data in chunk.iterrows():
            num_events=len(session_data.events)
            sessions+=([session_data.session]*num_events)
            for event in session_data.events:
                aids.append(event['aid'])
                tss.append(event['ts']/1000)
                types.append(type2id[event['type']])
    return pd.DataFrame(data={'session':sessions,'aid':aids,'ts':tss,'type':types})

In [30]:
test_df=json_df('../data/rawData/test.jsonl')

In [31]:
test_df.ts.head()

0    1.661724e+09
1    1.661724e+09
2    1.661724e+09
3    1.661724e+09
4    1.661724e+09
Name: ts, dtype: float64

In [32]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6928123 entries, 0 to 6928122
Data columns (total 4 columns):
 #   Column   Dtype  
---  ------   -----  
 0   session  int64  
 1   aid      int64  
 2   ts       float64
 3   type     int64  
dtypes: float64(1), int64(3)
memory usage: 211.4 MB


In [ ]:
train_df=json_df('../data/rawData/train.jsonl')
test_df=json_df('../data/rawData/test.jsonl')

train_df.type=train_df.type.astype(np.uint8)
test_df.type=test_df.type.astype(np.uint8)

train_df.to_parquet('../data/processData/train_parquet.parquet',index=False)
test_df.to_parquet('../data/processData/test_parquet.parquet',index=False)

train_df.to_csv('../data/processData/train.csv',index=False)
test_df.to_csv('../data/processData/test.csv',index=False)